In [ ]:
# Behavioral Validation (γ) — Gemma-3 27B IT (bf16)
# γ Behavioral Validation — Cell D near-inertness at behavioral level
# 4 orderings: A_imp_desc, B_imp_desc, C_imp_asc, D_imp_asc
# 5 ratios: 5%, 10%, 20%, 30%, 50%
# Measurements per trial: argmax_token, correct_logit, second_highest_logit, correct_rank
# Single stimulus set (seed 2026, 120 trials) — no multi-seed

# ── Cell 1 FIRST: Drive mount (user instruction) ──
from google.colab import drive, runtime
drive.mount('/content/drive')

!pip install -q nnsight transformers accelerate scipy pandas scikit-learn

import os
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
from huggingface_hub import login as _hf_login
_hf_login(token=os.environ['HF_TOKEN'])
print('HF_TOKEN set + huggingface_hub.login() called.')

import gc, math, random, time, traceback, json
from collections import defaultdict
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from nnsight import LanguageModel

PATCH_BATCH_SIZE = 32

MODEL_SPECS = [
    ('gghfez/gemma-3-27b-novision', 'gemma-3-27b-it'),
]

RATIOS = [0.05, 0.10, 0.20, 0.30, 0.50]
CELLS = ['A', 'B', 'C', 'D']  # 4 cells in paper

DRIVE_BASE = '/content/drive/MyDrive/WCC/models'
OUTPUT_PHASE = '38_behavioral'

CREL_PAIRS = [('SAME','OPP'),('SAME','MORE'),('SAME','LESS'),
              ('OPP','MORE'),('OPP','LESS'),('MORE','LESS')]
ATTR_DIMS = ['P', 'Q']
ATTR_P = ['bold', 'cautious']
ATTR_Q = ['complex', 'simple']

TEMPLATE_SAMEOPP = """Definitions:\n- {word_a}: something that is {attr1} and {attr2}\n- {word_b} is {relation}\n\nQ: Is {word_b} {choice1} or {choice2}? (Answer EXACTLY ONE WORD)\nA:"""
TEMPLATE_COMPARE = """Definitions:\n- {word_a}: something that is {attr1} and {attr2}\n- {word_b} is {relation}\n\nQ: Which is more {attr_asked}, {word_a} or {word_b}? (Answer EXACTLY ONE WORD)\nA:"""


def log(msg, flush=True):
    print(f'[{datetime.now().strftime("%H:%M:%S")}] {msg}', flush=flush)


def get_v(obj):
    return obj.value if hasattr(obj, 'value') else obj


def print_vram(prefix=''):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1024**3
        print(f'{prefix} VRAM: {a:.2f} GB', flush=True)


def load_model(model_id):
    log(f'Loading {model_id}')
    model = LanguageModel(model_id, dtype=torch.bfloat16, device_map='auto')
    cfg_m = model.config
    text_cfg = getattr(cfg_m, 'text_config', cfg_m)
    arch = {
        'num_layers': getattr(text_cfg, 'num_hidden_layers', getattr(text_cfg, 'n_layer', None)),
        'num_heads': text_cfg.num_attention_heads,
        'head_dim': getattr(text_cfg, 'head_dim', None) or (text_cfg.hidden_size // text_cfg.num_attention_heads),
        'hidden_size': text_cfg.hidden_size,
    }
    arch['total_heads'] = arch['num_layers'] * arch['num_heads']
    log(f'  Arch: {arch["num_layers"]}L x {arch["num_heads"]}H, total={arch["total_heads"]}')
    print_vram('[after load]')
    return model, arch


def build_prompts_and_answers(trials_df):
    """Build per-trial prompt + correct answer text (gold token string)."""
    p_pos, p_neg = ATTR_P
    q_pos, q_neg = ATTR_Q
    trial_prompts, trial_answers = {}, {}
    for _, row in trials_df.iterrows():
        tid, w1, w2 = row['trial_id'], row['word_a'], row['word_b']
        crel, attr_dim = row['crel'], row['attr_dim']
        correct = str(row['correct_answer'])
        if crel == 'SAME':
            prompt = TEMPLATE_SAMEOPP.format(word_a=w1, word_b=w2, attr1=p_pos, attr2=q_pos,
                relation=f'the same as {w1}',
                choice1=p_pos if attr_dim=='P' else q_pos,
                choice2=p_neg if attr_dim=='P' else q_neg)
        elif crel == 'OPP':
            prompt = TEMPLATE_SAMEOPP.format(word_a=w1, word_b=w2, attr1=p_pos, attr2=q_pos,
                relation=f'the opposite of {w1}',
                choice1=p_pos if attr_dim=='P' else q_pos,
                choice2=p_neg if attr_dim=='P' else q_neg)
        elif crel == 'MORE':
            prompt = TEMPLATE_COMPARE.format(word_a=w1, word_b=w2, attr1=p_pos, attr2=q_pos,
                relation=f'more/greater than {w1} in all respects',
                attr_asked=p_pos if attr_dim=='P' else q_pos)
        elif crel == 'LESS':
            prompt = TEMPLATE_COMPARE.format(word_a=w1, word_b=w2, attr1=p_pos, attr2=q_pos,
                relation=f'less/smaller than {w1} in all respects',
                attr_asked=p_pos if attr_dim=='P' else q_pos)
        else:
            continue
        trial_prompts[tid] = prompt
        trial_answers[tid] = correct
    return trial_prompts, trial_answers


def get_first_token_id(tokenizer, text):
    """Tokenize with leading space — the model generates next-token AFTER 'A:' which
    is typically the space-prefixed variant (e.g., ' bold' in Llama/Qwen/GPT-J BPE,
    '▁bold' in SentencePiece). Without the leading space, ID may not match what the
    model emits → argmax comparison systematically fails."""
    ids = tokenizer.encode(' ' + str(text), add_special_tokens=False)
    return ids[0] if ids else None


def get_first_token_id_bare(tokenizer, text):
    """BARE tokenization (NO leading space) — audit Q2 companion.
    This is the legacy bare-token path used in the prospect_v2 / dose_response_v2 outputs; we re-evaluate it below alongside the canonical normalized path for cross-validation.
    We record logits at this token in parallel to the correct space-prefixed token
    so we can later compute per-trial correlation between 'wrong-token delta' and
    'right-token delta', i.e., validate whether the paper's reported patching effects
    are a reliable proxy for the correct-token effects."""
    ids = tokenizer.encode(str(text), add_special_tokens=False)
    return ids[0] if ids else None


def build_4_orderings(cell_df, imp_col='rsa_max', pert_col='perturbation_l2'):
    """Build 4 orderings for γ: A_imp_desc, B_imp_desc, C_imp_asc, D_imp_asc.
    Uses existing cell classification from 20_scoring/cell_classification.csv."""
    orderings = {}
    for cell_letter, order_direction in [('A', 'desc'), ('B', 'desc'), ('C', 'asc'), ('D', 'asc')]:
        heads = [(int(r['layer']), int(r['head'])) for _, r in cell_df[cell_df['cell'] == cell_letter].iterrows()]
        imp_map = {(int(r['layer']), int(r['head'])): r[imp_col] for _, r in cell_df.iterrows()}
        if order_direction == 'desc':
            heads.sort(key=lambda lh: -imp_map[lh])
            ordering_name = f'{cell_letter}_imp_desc'
        else:
            heads.sort(key=lambda lh: imp_map[lh])
            ordering_name = f'{cell_letter}_imp_asc'
        orderings[ordering_name] = heads
    return orderings


def extract_trial_metrics(logits_row, correct_token_id):
    """Given (vocab,) logits, extract behavioral metrics for one trial.
    Returns: dict with argmax_token_id, correct_logit, second_highest_logit, correct_rank, argmax_eq_correct.
    """
    argmax_id = int(np.argmax(logits_row))
    correct_logit = float(logits_row[correct_token_id])
    # Second-highest: max over all except correct
    mask = np.ones_like(logits_row, dtype=bool)
    mask[correct_token_id] = False
    second_logit = float(logits_row[mask].max())
    # Rank: position of correct_token in sorted logits (1 = highest)
    rank = int((logits_row > correct_logit).sum()) + 1
    # +1 for ties above (strict >) then +1 for 1-indexed → if no ties, top rank = 1
    return {
        'argmax_token_id': argmax_id,
        'correct_logit': correct_logit,
        'second_highest_logit': second_logit,
        'correct_rank': rank,
        'argmax_eq_correct': int(argmax_id == correct_token_id),
    }


@torch.no_grad()
def get_batched_clean_logits(model, prompts):
    """Clean forward pass for B prompts. Returns (B, vocab) numpy."""
    with model.trace(prompts) as tracer:
        logits = model.output.logits[:, -1, :].save()
    return get_v(logits).detach().float().cpu().numpy()


@torch.no_grad()
def get_batched_group_logits(model, arch, prompts, source_vecs, patch_heads):
    """Patch same group of heads across B different (prompt, source_vec) pairs.
    Returns (B, vocab) numpy."""
    H, D = arch['num_heads'], arch['head_dim']
    B = len(prompts)
    sorted_patches = sorted(patch_heads, key=lambda lh: lh[0])
    patches_by_layer = defaultdict(list)
    for l, h in sorted_patches:
        patches_by_layer[l].append(h)
    src_stack = torch.tensor(np.stack(source_vecs), dtype=torch.bfloat16, device='cuda')
    with model.trace(prompts) as tracer:
        for layer_idx in sorted(patches_by_layer.keys()):
            for h_idx in patches_by_layer[layer_idx]:
                start = h_idx * D
                end = start + D
                model.model.layers[layer_idx].self_attn.o_proj.input[:, -1, start:end] = src_stack[:, layer_idx, h_idx]
        logits = model.output.logits[:, -1, :].save()
    return get_v(logits).detach().float().cpu().numpy()


def build_patch_trials(trials_df, all_vecs, tokenizer, trial_prompts, trial_answers):
    """Build per-trial patching specs (source donor + target recipient).
    For each target trial, pick source from same pair_id but different crel (same as prospect)."""
    patch_trials = []
    for sc, tc in CREL_PAIRS:
        for pid in sorted(trials_df['pair_id'].unique()):
            for ad in ATTR_DIMS:
                st, tt = f'{sc}_{ad}_{pid}', f'{tc}_{ad}_{pid}'
                if st not in all_vecs or tt not in trial_prompts:
                    continue
                tok = get_first_token_id(tokenizer, trial_answers[tt])
                tok_bare = get_first_token_id_bare(tokenizer, trial_answers[tt])  # Q2 audit
                if tok is None:
                    continue
                patch_trials.append({
                    'src_tid': st, 'tgt_tid': tt,
                    'crel_pair': f'{sc}-{tc}',
                    'pair_id': pid,
                    'attr_dim': ad,
                    'correct_token_id': tok,              # space-prefixed (correct)
                    'correct_token_id_bare': tok_bare,    # bare-token (legacy convention — for cross-validation only)
                    'correct_answer': trial_answers[tt],
                    'prompt': trial_prompts[tt],
                })
    return patch_trials


def run_behavioral_for_model(model_id, model_short, resume=True):
    """Main behavioral validation loop for one model."""
    drive_dir = f'{DRIVE_BASE}/{model_short}'
    output_dir = f'{drive_dir}/{OUTPUT_PHASE}'
    os.makedirs(output_dir, exist_ok=True)
    per_trial_path = f'{output_dir}/behavioral_per_trial.csv'
    aggregate_path = f'{output_dir}/behavioral_aggregate.csv'
    config_path = f'{output_dir}/config.json'

    # Resume check
    if resume and os.path.exists(aggregate_path):
        existing = pd.read_csv(aggregate_path)
        expected = len(CELLS) * len(RATIOS)  # 4 × 5 = 20
        if len(existing) >= expected:
            log(f'SKIP {model_short}: aggregate has {len(existing)} rows (>= {expected})')
            return True

    # Load model
    try:
        model, arch = load_model(model_id)
        tokenizer = model.tokenizer
        total_heads = arch['total_heads']
    except Exception as e:
        log(f'FAILED to load {model_short}: {type(e).__name__}: {str(e)[:200]}')
        traceback.print_exc()
        return False

    # Load existing data (baseline stimulus set, single 120-trial)
    try:
        vec_path = f'{drive_dir}/10_collection/activation_vectors.npz'
        trials_path = f'{drive_dir}/10_collection/trial_definitions.csv'
        cell_class_path = f'{drive_dir}/20_scoring/cell_classification.csv'
        log(f'  Loading vectors: {vec_path}')
        npz = np.load(vec_path)
        all_vecs = {k: npz[k] for k in npz.files}
        npz.close()
        trials_df = pd.read_csv(trials_path)
        cell_df = pd.read_csv(cell_class_path)
        log(f'  Loaded {len(all_vecs)} vectors, {len(trials_df)} trials, {len(cell_df)} head classifications')
    except Exception as e:
        log(f'FAILED data load for {model_short}: {type(e).__name__}: {str(e)[:200]}')
        return False

    # Build prompts + patch trials
    trial_prompts, trial_answers = build_prompts_and_answers(trials_df)
    patch_trials = build_patch_trials(trials_df, all_vecs, tokenizer, trial_prompts, trial_answers)
    log(f'  Built {len(patch_trials)} patch trials (expected 120)')

    # Orderings (A/B/C/D)
    orderings = build_4_orderings(cell_df)
    log(f'  Orderings: ' + ', '.join(f'{k}={len(v)}' for k, v in orderings.items()))

    # Clean baseline (120 trials, unique prompts)
    log(f'  Computing clean baseline ...')
    unique_tgts = sorted(set(pt['tgt_tid'] for pt in patch_trials))
    clean_cache = {}
    for bi in range(0, len(unique_tgts), PATCH_BATCH_SIZE):
        batch_tids = unique_tgts[bi:bi + PATCH_BATCH_SIZE]
        batch_prompts = [trial_prompts[t] for t in batch_tids]
        try:
            logits_b = get_batched_clean_logits(model, batch_prompts)
            for j, tid in enumerate(batch_tids):
                clean_cache[tid] = logits_b[j]
        except Exception as e:
            log(f'    clean batch {bi} fail: {type(e).__name__}, fallback per-trial')
            for tid in batch_tids:
                with model.trace([trial_prompts[tid]]) as tr:
                    lo = model.output.logits[0, -1, :].save()
                clean_cache[tid] = get_v(lo).detach().float().cpu().numpy()

    # Patching loop: 4 cells × 5 ratios
    per_trial_records = []
    aggregate_records = []

    for cell_letter in CELLS:
        ordering_name = f'{cell_letter}_imp_desc' if cell_letter in ('A', 'B') else f'{cell_letter}_imp_asc'
        glist = orderings[ordering_name]
        for ratio in RATIOS:
            t_start = time.time()
            k = max(1, math.ceil(total_heads * ratio))
            k_actual = min(k, len(glist))
            heads_k = glist[:k_actual]

            # Batched patching
            trial_records = []
            for bi in range(0, len(patch_trials), PATCH_BATCH_SIZE):
                batch = patch_trials[bi:bi + PATCH_BATCH_SIZE]
                prompts_b = [pt['prompt'] for pt in batch]
                src_vecs_b = [all_vecs[pt['src_tid']] for pt in batch]
                try:
                    logits_b = get_batched_group_logits(model, arch, prompts_b, src_vecs_b, heads_k)
                except Exception as e:
                    log(f'    batch {bi} fail: {type(e).__name__}: {str(e)[:100]}')
                    for pt in batch:
                        trial_records.append({
                            'model': model_short, 'cell': cell_letter, 'ratio': ratio,
                            'crel_pair': pt['crel_pair'], 'pair_id': pt['pair_id'], 'attr_dim': pt['attr_dim'],
                            'trial_id': pt['tgt_tid'],
                            'error': type(e).__name__,
                        })
                    continue
                for j, pt in enumerate(batch):
                    tid = pt['tgt_tid']
                    correct_tok = pt['correct_token_id']
                    correct_tok_bare = pt['correct_token_id_bare']  # Q2 audit
                    clean_logits = clean_cache[tid]
                    patched_logits = logits_b[j]
                    clean_m = extract_trial_metrics(clean_logits, correct_tok)
                    patched_m = extract_trial_metrics(patched_logits, correct_tok)
                    # Q2 audit: bare-token logits (same trial, WRONG token)
                    clean_logit_bare  = float(clean_logits[correct_tok_bare])  if correct_tok_bare is not None else float('nan')
                    patched_logit_bare = float(patched_logits[correct_tok_bare]) if correct_tok_bare is not None else float('nan')
                    trial_records.append({
                        'model': model_short, 'cell': cell_letter, 'ratio': ratio,
                        'crel_pair': pt['crel_pair'], 'pair_id': pt['pair_id'], 'attr_dim': pt['attr_dim'],
                        'trial_id': tid, 'correct_answer': pt['correct_answer'], 'correct_token_id': correct_tok,
                        'clean_argmax_token_id': clean_m['argmax_token_id'],
                        'clean_correct_logit': clean_m['correct_logit'],
                        'clean_second_logit': clean_m['second_highest_logit'],
                        'clean_correct_rank': clean_m['correct_rank'],
                        'clean_correct': clean_m['argmax_eq_correct'],
                        'patched_argmax_token_id': patched_m['argmax_token_id'],
                        'patched_correct_logit': patched_m['correct_logit'],
                        'patched_second_logit': patched_m['second_highest_logit'],
                        'patched_correct_rank': patched_m['correct_rank'],
                        'patched_correct': patched_m['argmax_eq_correct'],
                        'delta_correct_logit': patched_m['correct_logit'] - clean_m['correct_logit'],
                        # Cross-validation: bare-token measurements (legacy-path replication)
                        'correct_token_id_bare': correct_tok_bare,
                        'clean_correct_logit_bare': clean_logit_bare,
                        'patched_correct_logit_bare': patched_logit_bare,
                        'delta_correct_logit_bare': patched_logit_bare - clean_logit_bare,
                    })
            per_trial_records.extend(trial_records)

            # Aggregate for this (cell, ratio)
            valid = [r for r in trial_records if 'error' not in r]
            n_valid = len(valid)
            # Initialize to NaN so log line never hits NameError
            clean_acc = float('nan')
            patched_acc = float('nan')
            if n_valid > 0:
                clean_acc = float(np.mean([r['clean_correct'] for r in valid]))
                patched_acc = float(np.mean([r['patched_correct'] for r in valid]))
                mean_delta_logit = float(np.mean([r['delta_correct_logit'] for r in valid]))
                clean_margins = [r['clean_correct_logit'] - r['clean_second_logit'] for r in valid]
                patched_margins = [r['patched_correct_logit'] - r['patched_second_logit'] for r in valid]
                mean_delta_margin = float(np.mean(np.array(patched_margins) - np.array(clean_margins)))
                median_patched_rank = float(np.median([r['patched_correct_rank'] for r in valid]))
                rank_deterioration = float(np.mean([r['patched_correct_rank'] > 1 for r in valid]))
                aggregate_records.append({
                    'model': model_short, 'cell': cell_letter, 'ratio': ratio,
                    'ordering': ordering_name, 'k': k, 'k_actual': k_actual, 'n_trials': n_valid,
                    'clean_accuracy': clean_acc, 'patched_accuracy': patched_acc,
                    'accuracy_drop': clean_acc - patched_acc,
                    'mean_delta_correct_logit': mean_delta_logit,
                    'mean_delta_margin': mean_delta_margin,
                    'median_patched_rank': median_patched_rank,
                    'rank_deterioration_rate': rank_deterioration,
                })
            else:
                aggregate_records.append({
                    'model': model_short, 'cell': cell_letter, 'ratio': ratio,
                    'ordering': ordering_name, 'k': k, 'k_actual': k_actual, 'n_trials': 0,
                    'error': 'no_valid_trials',
                })

            # Incremental save after each (cell, ratio)
            pd.DataFrame(per_trial_records).to_csv(per_trial_path, index=False)
            pd.DataFrame(aggregate_records).to_csv(aggregate_path, index=False)
            log(f'    [{model_short}] {cell_letter}/{ratio:.2f}: patched_acc={patched_acc:.3f} drop={clean_acc-patched_acc:+.3f} ({time.time()-t_start:.0f}s)')

    # Save config
    with open(config_path, 'w') as f:
        json.dump({
            'experiment': 'gamma_behavioral_validation',
            'model': model_short,
            'model_id': model_id,
            'ratios': RATIOS, 'cells': CELLS,
            'orderings': {c: (f'{c}_imp_desc' if c in ('A', 'B') else f'{c}_imp_asc') for c in CELLS},
            'n_trials': len(patch_trials),
            'total_heads': total_heads, 'arch': {k: v for k, v in arch.items() if k != 'dtype'},
            'single_stimulus_set_seed': 2026,
            'created_at': datetime.now().isoformat(),
        }, f, indent=2)
    log(f'DONE: {model_short}')

    # Unload model
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    return True


print('Cell 0 ready. Functions defined.')


In [ ]:
# ── Cell 2: Model loop + auto-unassign ──

all_success = True
for MODEL_ID, MODEL_SHORT in MODEL_SPECS:
    log(f'=' * 60)
    log(f'START: {MODEL_SHORT}')
    t0 = time.time()
    try:
        ok = run_behavioral_for_model(MODEL_ID, MODEL_SHORT, resume=True)
        log(f'  {"OK" if ok else "FAIL"}: {MODEL_SHORT} in {(time.time()-t0)/60:.1f} min')
        if not ok:
            all_success = False
    except Exception as e:
        log(f'FAILED: {MODEL_SHORT} — {type(e).__name__}: {str(e)[:300]}')
        traceback.print_exc()
        all_success = False

log(f'=' * 60)
log(f'ALL MODELS PROCESSED. overall_success={all_success}')

# Auto-unassign runtime (user instruction: kill after all models done)
if all_success:
    log('Unassigning Colab runtime (all models complete).')
    runtime.unassign()
else:
    log('Some models failed; keeping runtime alive for inspection.')
